# Análise de Preços do Ponto de Função - Inventário 2026

**SEPLAG/MT - Comissão de Inventário e Avaliação de Bens Intangíveis**

Este notebook faz a parte de engenharia e ciência de dados da pesquisa de preços. Ele **não escolhe o preço**: produz as tabelas, os grupos e os gráficos para a decisão da presidência da comissão.

| Etapa | O que faz |
|---|---|
| 1 | Carrega os 9 preços Java em ponto de função (2025-2026), o Contrato 014/2022/SEPLAG e os 13 preços da pesquisa de 2025 |
| 2 | Estatística descritiva dos preços originais e dos atualizados pelo IPCA |
| 3 | Explica e mostra a atualização monetária, preço a preço |
| 4 | Agrupa os preços em 3 grupos (K-Means), com os preços originais e com os atualizados |
| 5 | Testa a sensibilidade dos grupos a mudanças na amostra |
| 6 | Compara 2025 com 2026, com e sem atualização monetária |
| 7 | Grava os resultados (JSON e Excel) e os gráficos em `graficos/` |

Para executar: `python executar.py --analise` na pasta `2026`, ou *Run All* no VS Code.

In [1]:
# ==========================================
# CÉLULA 0: Bibliotecas, estilo e funções auxiliares
# ==========================================
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

PASTA = Path.cwd()                      # 5. Análise Estatística
RAIZ = PASTA.parent                     # 2.3 Mapa Comparativo de Preços
GRAF = PASTA / 'graficos'
GRAF.mkdir(exist_ok=True)

CFG = json.loads((RAIZ / 'config_pesquisa.json').read_text(encoding='utf-8'))
K, SEED, NINIT = CFG['kmeans']['grupos'], CFG['kmeans']['random_state'], CFG['kmeans']['n_init']

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.25, 'text.parse_math': False})
COR_GRUPO = {1: '#1b9e77', 2: '#e6a118', 3: '#d95f02'}
NOME_GRUPO = {1: 'Grupo 1 (Entrada)', 2: 'Grupo 2 (Intermediário)', 3: 'Grupo 3 (Premium)'}
COR_ORIG, COR_ATU = '#7a8794', '#0b4f9e'
COR_CONTRATO, COR_2025 = '#2e7d32', '#8e24aa'

def brl(v, casas=2):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return '-'
    s = f'{v:,.{casas}f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
    return f'R$ {s}'

def pct(v, casas=2):
    return f'{v:.{casas}f}%'.replace('.', ',')

eixo_brl = mtick.FuncFormatter(lambda x, _: brl(x, 0))

def salvar(fig, nome):
    fig.savefig(GRAF / nome, bbox_inches='tight', facecolor='white')
    plt.show()

def kmeans_grupos(df, coluna, rotulo='Rotulo'):
    # Agrupa os preços de 'coluna' em K grupos e numera os grupos do mais barato (1) ao mais caro (K)
    d = df.copy()
    modelo = KMeans(n_clusters=K, random_state=SEED, n_init=NINIT).fit(d[[coluna]])
    ordem = np.argsort(modelo.cluster_centers_.ravel())
    mapa = {int(c): i + 1 for i, c in enumerate(ordem)}
    d['Grupo'] = [mapa[int(c)] for c in modelo.labels_]
    sil = silhouette_score(d[[coluna]], modelo.labels_) if d['Grupo'].nunique() > 1 and len(d) > K else float('nan')
    linhas = []
    for g in range(1, K + 1):
        x = d[d['Grupo'] == g].sort_values(coluna)
        linhas.append({'Grupo': NOME_GRUPO[g], 'Qtde de preços': len(x),
                       'Preços do grupo': '; '.join(f"{r[rotulo]}: {brl(r[coluna])}" for _, r in x.iterrows()),
                       'Mínimo': x[coluna].min(), 'Mediana': x[coluna].median(), 'Centroide (média)': x[coluna].mean(), 'Máximo': x[coluna].max()})
    return d, pd.DataFrame(linhas), sil

def mostra(df, moeda=(), pcts=()):
    f = {c: brl for c in moeda}
    f.update({c: pct for c in pcts})
    display(df.style.format(f, na_rep='-').hide(axis='index'))

## 1. Dados de entrada

* **Amostra 2026:** 9 preços de serviços em Java medidos em ponto de função, coletados na API oficial do Compras.gov.br (pasta `4. Mapa Comparativo`).
* **Contrato 014/2022/SEPLAG:** objeto de referência. Preço vigente após o 5º apostilamento (`config_pesquisa.json`).
* **Pesquisa 2025:** os 13 preços usados no inventário de 2025, nos valores originais e atualizados até 10/2025.
* **IPCA:** série 433 do Banco Central, gravada em `2. Coleta de Dados/Dados_Brutos`.

In [2]:
# ==========================================
# CÉLULA 1: Carregamento dos dados
# ==========================================
mapa = pd.read_csv(RAIZ / '4. Mapa Comparativo' / 'Precos_Ponto_Funcao_2025-2026.csv', sep=';', encoding='utf-8-sig')
a = mapa[mapa['classificacao'].str.startswith('AMOSTRA A')].copy()
apelido = {1: 'Casa da Moeda', 2: 'SESED-RN', 3: 'Marinha', 4: 'UFCA', 5: 'MP-PA', 6: 'INCRA/Serpro', 7: 'TJ-PA', 8: 'MDS', 9: 'SMED Salvador'}
d26 = pd.DataFrame({
    'Ref': a['id'].astype(int),
    'Rotulo': [f'Ref {i} {apelido.get(i, "")}'.strip() for i in a['id'].astype(int)],
    'Órgão': a['uasg_nome'].str.title(),
    'Modalidade': a['modalidade_nome'],
    'Data da compra': pd.to_datetime(a['data_compra']).dt.strftime('%d/%m/%Y'),
    'Preço original': a['preco_unitario_pf'].astype(float),
    'Mês da compra': a['mes_base_preco'],
    'Mês final': a['data_base_atualizacao'],
    'Fator IPCA': a['fator_ipca'].astype(float),
    'Preço atualizado': a['preco_atualizado_pf'].astype(float),
    'Dentro de 13 meses': a['dentro_13_meses'],
    'Disputa de preço': a['disputa_de_preco'],
}).reset_index(drop=True)
d26['IPCA acumulado'] = (d26['Fator IPCA'] - 1) * 100
d26['Aumento (R$)'] = d26['Preço atualizado'] - d26['Preço original']
DATA_BASE = d26['Mês final'].iloc[0]

ct = CFG['contrato_referencia']
CONTRATO = ct['preco_vigente_pf']
PRECO_2025 = CFG['inventario_2025']['preco_pf_adotado']

ipca = json.loads((RAIZ / '2. Coleta de Dados' / 'Dados_Brutos' / 'ipca_serie_433_bcb.json').read_text(encoding='utf-8'))['dados']
ipca = {(int(x['data'][6:]), int(x['data'][3:5])): float(x['valor']) for x in ipca}
def fator_ipca(ano, mes):
    f = 1.0
    for (aa, mm), v in sorted(ipca.items()):
        if (aa, mm) >= (ano, mes):
            f *= 1 + v / 100
    return f

if ct.get('mes_base_preco_vigente'):
    aa, mm = map(int, ct['mes_base_preco_vigente'].split('-'))
    CONTRATO_ATU = CONTRATO * fator_ipca(aa, mm)
else:
    CONTRATO_ATU = CONTRATO   # mês-base do 5º apostilamento ainda não informado: sem correção

p25 = pd.read_csv(RAIZ / '4. Mapa Comparativo' / 'Precos_Pesquisa_2025.csv', sep=';')
FATOR_2025_2026 = fator_ipca(2025, 11)          # IPCA de 11/2025 até o último mês publicado
d25 = pd.DataFrame({
    'Ref': p25['id'],
    'Rotulo': [f'P{i}' for i in p25['id']],
    'Fonte': p25['fonte'], 'Órgão': p25['orgao'],
    'Preço original': p25['preco_original_pf'],
    'Atualizado até 10/2025': p25['preco_atualizado_10_2025'],
})
d25[f'Atualizado até {DATA_BASE}'] = (d25['Atualizado até 10/2025'] * FATOR_2025_2026).round(2)

print(f'Amostra 2026: {len(d26)} preços | data-base do IPCA: {DATA_BASE}')
print(f'Contrato 014/2022/SEPLAG: {brl(CONTRATO)} ({ct["ato_vigente"]}, {ct["processo_ato_vigente"]})')
print(f'Pesquisa 2025: {len(d25)} preços | IPCA de 11/2025 a {DATA_BASE}: {pct((FATOR_2025_2026 - 1) * 100)}')
mostra(d26[['Ref', 'Órgão', 'Modalidade', 'Data da compra', 'Preço original', 'Preço atualizado']], moeda=['Preço original', 'Preço atualizado'])

Amostra 2026: 9 preços | data-base do IPCA: 10/2026
Contrato 014/2022/SEPLAG: R$ 544,53 (5º Termo de Apostilamento, SEPLAG-PRO-2026/09621)
Pesquisa 2025: 13 preços | IPCA de 11/2025 a 10/2026: 4,63%


Ref,Órgão,Modalidade,Data da compra,Preço original,Preço atualizado
1,Casa Da Moeda Do Brasil/Mf,Pregão - Eletrônico,17/02/2025,"R$ 555,00","R$ 601,41"
2,Ern-Secret.Da Segurança Publ. E Defesa Social,Inexigibilidade,09/06/2025,"R$ 565,15","R$ 597,00"
3,Pagadoria De Pessoal Da Marinha - Com Marinha,Pregão - Eletrônico,15/08/2025,"R$ 1.060,00","R$ 1.114,16"
4,Universidade Federal Do Cariri,Pregão - Eletrônico,14/11/2025,"R$ 735,00","R$ 769,02"
5,Ministério Público Do Estado Do Pará,Inexigibilidade,10/12/2025,"R$ 750,00","R$ 783,30"
6,Incra-Sede/Df,Dispensa,23/12/2025,"R$ 2.192,73","R$ 2.290,10"
7,Tribunal De Justiça Do Estado Do Pará,Pregão - Eletrônico,07/01/2026,"R$ 1.264,22","R$ 1.316,01"
8,Subsecretaria De Assuntos Administrativos,Pregão - Eletrônico,26/02/2026,"R$ 538,00","R$ 558,20"
9,Secretaria Municipal Da Educação De Salvador,Inexigibilidade,20/08/2026,"R$ 785,00","R$ 790,01"


## 2. Estatística descritiva: preços originais e atualizados

**Preço original** é o valor homologado na data da compra. **Preço atualizado** é o mesmo valor corrigido pelo IPCA até a data-base.
O teste de outliers usa o intervalo interquartil (IQR): é outlier o preço acima de Q3 + 1,5 × IQR ou abaixo de Q1 − 1,5 × IQR.

In [3]:
# ==========================================
# CÉLULA 2: Estatística descritiva e teste de outliers (IQR)
# ==========================================
def descritiva(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75); iqr = q3 - q1
    li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return {'Quantidade': len(s), 'Mínimo': s.min(), 'Primeiro quartil (Q1)': q1, 'Mediana': s.median(), 'Média': s.mean(),
            'Terceiro quartil (Q3)': q3, 'Máximo': s.max(), 'Desvio padrão': s.std(), 'Coeficiente de variação': s.std() / s.mean() * 100,
            'Limite inferior IQR': li, 'Limite superior IQR': ls, 'Outliers': [float(x) for x in s[(s < li) | (s > ls)]]}

desc = {'Preço original': descritiva(d26['Preço original']), 'Preço atualizado': descritiva(d26['Preço atualizado'])}
tab_desc = pd.DataFrame(desc)
tab_view = tab_desc.copy().astype(object)
for c in tab_view.columns:
    for i in tab_view.index:
        v = tab_view.loc[i, c]
        if i == 'Quantidade': tab_view.loc[i, c] = str(v)
        elif i == 'Coeficiente de variação': tab_view.loc[i, c] = pct(v, 1)
        elif i == 'Outliers': tab_view.loc[i, c] = ', '.join(brl(x) for x in v) or 'nenhum'
        else: tab_view.loc[i, c] = brl(v)
display(tab_view)
outlier_refs = d26.loc[d26['Preço atualizado'] > desc['Preço atualizado']['Limite superior IQR'], 'Ref'].tolist()
print('Referências apontadas como outlier (preço atualizado):', outlier_refs)

,Preço original,Preço atualizado
Quantidade,9,9
Mínimo,"R$ 538,00","R$ 558,20"
Primeiro quartil (Q1),"R$ 565,15","R$ 601,41"
Mediana,"R$ 750,00","R$ 783,30"
Média,"R$ 938,34","R$ 979,91"
Terceiro quartil (Q3),"R$ 1.060,00","R$ 1.114,16"
Máximo,"R$ 2.192,73","R$ 2.290,10"
Desvio padrão,"R$ 529,31","R$ 551,65"
Coeficiente de variação,"56,4%","56,3%"
Limite inferior IQR,"R$ -177,13","R$ -167,72"


Referências apontadas como outlier (preço atualizado): [6]


In [4]:
# ==========================================
# CÉLULA 3: Gráfico G01 - distribuição dos preços (original x atualizado)
# Eixo vertical = quantidade de preços em cada faixa de R$ 250
# ==========================================
faixas = np.arange(500, 2501, 250)
fig, eixos = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
ymax = max(np.histogram(d26[c], bins=faixas)[0].max() for c in ['Preço original', 'Preço atualizado']) + 1
for ax, col, cor, tit in [(eixos[0], 'Preço original', COR_ORIG, 'Preços originais (sem atualização)'),
                          (eixos[1], 'Preço atualizado', COR_ATU, f'Preços atualizados pelo IPCA até {DATA_BASE}')]:
    n, b, barras = ax.hist(d26[col], bins=faixas, color=cor, edgecolor='white', alpha=0.9)
    for qt, x0 in zip(n, b[:-1]):
        if qt:
            ax.text(x0 + 125, qt + 0.08, f'{int(qt)}', ha='center', fontweight='bold')
    ax.axvline(d26[col].median(), color='black', ls='-', lw=1.6, label=f'Mediana: {brl(d26[col].median())}')
    ax.axvline(d26[col].mean(), color='black', ls=':', lw=1.6, label=f'Média: {brl(d26[col].mean())}')
    ax.axvline(CONTRATO if col == 'Preço original' else CONTRATO_ATU, color=COR_CONTRATO, ls='--', lw=1.6,
               label=f'Contrato 014/2022/SEPLAG: {brl(CONTRATO)}')
    ax.set_title(tit); ax.set_xlabel('Preço do ponto de função (faixas de R$ 250)')
    ax.xaxis.set_major_formatter(eixo_brl); ax.set_xticks(faixas); ax.tick_params(axis='x', rotation=45)
    ax.yaxis.set_major_locator(mtick.MaxNLocator(integer=True)); ax.set_ylim(0, ymax)
    ax.legend(loc='upper right', fontsize=8.5)
eixos[0].set_ylabel('Quantidade de preços')
fig.suptitle('G01 - Distribuição dos 9 preços Java por faixa de valor', fontweight='bold', y=1.02)
salvar(fig, 'G01_distribuicao_original_x_atualizado.png')

In [5]:
# ==========================================
# CÉLULA 4: Gráfico G02 - boxplot (caixa = 50% centrais dos preços; linha preta = mediana)
# ==========================================
fig, eixos = plt.subplots(2, 1, figsize=(13, 5.6), sharex=True)
for ax, col, cor, tit in [(eixos[0], 'Preço original', COR_ORIG, 'Preços originais'), (eixos[1], 'Preço atualizado', COR_ATU, f'Preços atualizados até {DATA_BASE}')]:
    dd = desc[col]
    ax.boxplot(d26[col], vert=False, widths=0.5, patch_artist=True, showfliers=False,
               boxprops=dict(facecolor=cor, alpha=0.25, edgecolor=cor), medianprops=dict(color='black', lw=2.2), whiskerprops=dict(color=cor), capprops=dict(color=cor))
    dentro = d26[d26[col] <= dd['Limite superior IQR']]
    fora = d26[d26[col] > dd['Limite superior IQR']]
    ax.scatter(dentro[col], [1] * len(dentro), s=55, color=cor, edgecolor='black', zorder=3, label='Preço dentro do limite')
    ax.scatter(fora[col], [1] * len(fora), s=90, color='#c62828', edgecolor='black', zorder=3, label='Outlier (acima do limite superior)')
    for _, r in fora.iterrows():
        ax.annotate(f"Ref {r['Ref']} {apelido[r['Ref']]}: {brl(r[col])}", (r[col], 1), xytext=(0, 18), textcoords='offset points', ha='center', fontsize=9, color='#c62828', fontweight='bold')
    for v, nome, dy in [(dd['Primeiro quartil (Q1)'], 'Q1', -34), (dd['Mediana'], 'Mediana', 30), (dd['Terceiro quartil (Q3)'], 'Q3', -34)]:
        ax.annotate(f'{nome}\n{brl(v)}', (v, 1), xytext=(0, dy), textcoords='offset points', ha='center', fontsize=8.5)
    ax.axvline(dd['Limite superior IQR'], color='#c62828', ls='--', lw=1.2)
    ax.text(dd['Limite superior IQR'], 1.4, f"  limite superior IQR: {brl(dd['Limite superior IQR'])}", color='#c62828', fontsize=8.5, va='center')
    ax.set_yticks([]); ax.set_ylim(0.45, 1.55); ax.set_title(tit, loc='left')
eixos[0].legend(loc='upper center', fontsize=8.5, ncol=2)
eixos[1].xaxis.set_major_formatter(eixo_brl); eixos[1].set_xlabel('Preço do ponto de função')
fig.suptitle('G02 - Boxplot: onde está a maior parte dos preços e qual preço passa do limite de outlier', fontweight='bold')
fig.tight_layout()
salvar(fig, 'G02_boxplot_original_x_atualizado.png')

/tmp/ipykernel_750/3976513562.py:7: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(d26[col], vert=False, widths=0.5, patch_artist=True, showfliers=False,
/tmp/ipykernel_750/3976513562.py:7: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(d26[col], vert=False, widths=0.5, patch_artist=True, showfliers=False,


## 3. Como funciona a atualização monetária pelo IPCA

1. Para cada preço, identifica-se o **mês da compra** (data em que o preço foi homologado).
2. Busca-se no Banco Central a variação mensal do IPCA de cada mês, **do mês da compra até a data-base** (último mês publicado).
3. Multiplicam-se os fatores mensais: **Fator = (1 + IPCA₁/100) × (1 + IPCA₂/100) × ... × (1 + IPCAₙ/100)**.
4. **Preço atualizado = Preço original × Fator.** O **IPCA acumulado** é (Fator − 1) × 100.

Exemplo: um preço de R$ 538,00 de fevereiro de 2026 multiplicado pelo fator 1,037545 (IPCA acumulado de 3,75% entre 02/2026 e 10/2026) resulta em R$ 558,20.
O método é o mesmo da ferramenta `Atualização_Monetária_IPCA.py` usada em 2025 e inclui o IPCA do próprio mês da compra.

In [6]:
# ==========================================
# CÉLULA 5: Tabela da atualização monetária e gráfico G03
# ==========================================
tab_ipca = d26[['Ref', 'Órgão', 'Preço original', 'Mês da compra', 'Mês final', 'IPCA acumulado', 'Fator IPCA', 'Preço atualizado', 'Aumento (R$)']].rename(
    columns={'Preço original': 'Preço original (R$/PF)', 'Mês da compra': 'Início da correção', 'Mês final': 'Fim da correção',
             'IPCA acumulado': 'IPCA acumulado no período', 'Fator IPCA': 'Fator de correção', 'Preço atualizado': 'Preço atualizado (R$/PF)',
             'Aumento (R$)': 'Diferença (R$)'})
display(tab_ipca.style.format({'Preço original (R$/PF)': brl, 'Preço atualizado (R$/PF)': brl, 'Diferença (R$)': brl,
                               'IPCA acumulado no período': pct, 'Fator de correção': '{:.6f}'}).hide(axis='index'))

o = d26.sort_values('Preço original').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(13, 5.8))
for i, r in o.iterrows():
    ax.plot([r['Preço original'], r['Preço atualizado']], [i, i], color='#b0bec5', lw=3, zorder=1)
    ax.scatter(r['Preço original'], i, s=80, color=COR_ORIG, zorder=2, label='Preço original' if i == 0 else None)
    ax.scatter(r['Preço atualizado'], i, s=80, color=COR_ATU, zorder=3, label=f'Preço atualizado até {DATA_BASE}' if i == 0 else None)
    ax.text(r['Preço atualizado'] + 25, i, f"{brl(r['Preço original'])} → {brl(r['Preço atualizado'])}  (+{pct(r['IPCA acumulado'])}, compra em {r['Mês da compra']})",
            va='center', fontsize=8.5)
ax.set_yticks(range(len(o))); ax.set_yticklabels([f"Ref {r['Ref']} - {apelido[r['Ref']]}" for _, r in o.iterrows()])
ax.xaxis.set_major_formatter(eixo_brl); ax.set_xlim(450, o['Preço atualizado'].max() * 1.55)
ax.set_xlabel('Preço do ponto de função'); ax.legend(loc='lower right')
ax.set_title('G03 - Efeito da atualização monetária em cada preço (do mês da compra até a data-base)')
salvar(fig, 'G03_atualizacao_ipca_por_preco.png')

Ref,Órgão,Preço original (R$/PF),Início da correção,Fim da correção,IPCA acumulado no período,Fator de correção,Preço atualizado (R$/PF),Diferença (R$)
1,Casa Da Moeda Do Brasil/Mf,"R$ 555,00",02/2025,10/2026,"8,36%",1.083626,"R$ 601,41","R$ 46,41"
2,Ern-Secret.Da Segurança Publ. E Defesa Social,"R$ 565,15",06/2025,10/2026,"5,64%",1.056356,"R$ 597,00","R$ 31,85"
3,Pagadoria De Pessoal Da Marinha - Com Marinha,"R$ 1.060,00",08/2025,10/2026,"5,11%",1.051094,"R$ 1.114,16","R$ 54,16"
4,Universidade Federal Do Cariri,"R$ 735,00",11/2025,10/2026,"4,63%",1.046284,"R$ 769,02","R$ 34,02"
5,Ministério Público Do Estado Do Pará,"R$ 750,00",12/2025,10/2026,"4,44%",1.044404,"R$ 783,30","R$ 33,30"
6,Incra-Sede/Df,"R$ 2.192,73",12/2025,10/2026,"4,44%",1.044404,"R$ 2.290,10","R$ 97,37"
7,Tribunal De Justiça Do Estado Do Pará,"R$ 1.264,22",01/2026,10/2026,"4,10%",1.040968,"R$ 1.316,01","R$ 51,79"
8,Subsecretaria De Assuntos Administrativos,"R$ 538,00",02/2026,10/2026,"3,75%",1.037545,"R$ 558,20","R$ 20,20"
9,Secretaria Municipal Da Educação De Salvador,"R$ 785,00",08/2026,10/2026,"0,64%",1.006381,"R$ 790,01","R$ 5,01"


## 4. Segmentação em grupos (K-Means): preços originais x atualizados

O K-Means separa os preços em 3 grupos pela distância entre eles. O valor representativo de cada grupo é o **centroide** (média dos preços do grupo); a tabela mostra também mínimo, mediana e máximo, e lista cada preço.
Os grupos são numerados do mais barato (Grupo 1) ao mais caro (Grupo 3). Parâmetros: `random_state = 42`, `n_init = 10`, iguais aos de 2025.

In [7]:
# ==========================================
# CÉLULA 6: K-Means nos preços originais e nos atualizados
# ==========================================
seg_o, res_o, sil_o = kmeans_grupos(d26, 'Preço original')
seg_a, res_a, sil_a = kmeans_grupos(d26, 'Preço atualizado')
moeda = ['Mínimo', 'Mediana', 'Centroide (média)', 'Máximo']
print(f'PREÇOS ORIGINAIS (índice de silhueta {sil_o:.2f})'); mostra(res_o, moeda=moeda)
print(f'PREÇOS ATUALIZADOS ATÉ {DATA_BASE} (índice de silhueta {sil_a:.2f})'); mostra(res_a, moeda=moeda)
mudou = seg_o.set_index('Ref')['Grupo'] != seg_a.set_index('Ref')['Grupo']
print('Preços que mudam de grupo com a atualização:', list(mudou[mudou].index) or 'nenhum')

PREÇOS ORIGINAIS (índice de silhueta 0.61)


Grupo,Qtde de preços,Preços do grupo,Mínimo,Mediana,Centroide (média),Máximo
Grupo 1 (Entrada),6,"Ref 8 MDS: R$ 538,00; Ref 1 Casa da Moeda: R$ 555,00; Ref 2 SESED-RN: R$ 565,15; Ref 4 UFCA: R$ 735,00; Ref 5 MP-PA: R$ 750,00; Ref 9 SMED Salvador: R$ 785,00","R$ 538,00","R$ 650,08","R$ 654,69","R$ 785,00"
Grupo 2 (Intermediário),2,"Ref 3 Marinha: R$ 1.060,00; Ref 7 TJ-PA: R$ 1.264,22","R$ 1.060,00","R$ 1.162,11","R$ 1.162,11","R$ 1.264,22"
Grupo 3 (Premium),1,"Ref 6 INCRA/Serpro: R$ 2.192,73","R$ 2.192,73","R$ 2.192,73","R$ 2.192,73","R$ 2.192,73"


PREÇOS ATUALIZADOS ATÉ 10/2026 (índice de silhueta 0.64)


Grupo,Qtde de preços,Preços do grupo,Mínimo,Mediana,Centroide (média),Máximo
Grupo 1 (Entrada),6,"Ref 8 MDS: R$ 558,20; Ref 2 SESED-RN: R$ 597,00; Ref 1 Casa da Moeda: R$ 601,41; Ref 4 UFCA: R$ 769,02; Ref 5 MP-PA: R$ 783,30; Ref 9 SMED Salvador: R$ 790,01","R$ 558,20","R$ 685,21","R$ 683,16","R$ 790,01"
Grupo 2 (Intermediário),2,"Ref 3 Marinha: R$ 1.114,16; Ref 7 TJ-PA: R$ 1.316,01","R$ 1.114,16","R$ 1.215,09","R$ 1.215,09","R$ 1.316,01"
Grupo 3 (Premium),1,"Ref 6 INCRA/Serpro: R$ 2.290,10","R$ 2.290,10","R$ 2.290,10","R$ 2.290,10","R$ 2.290,10"


Preços que mudam de grupo com a atualização: nenhum


In [8]:
# ==========================================
# CÉLULA 7: Gráfico G04 - cada preço em uma barra, na cor do seu grupo
# A mesma referência fica na mesma linha nos dois painéis, para comparar original x atualizado
# ==========================================
ordem = d26.sort_values('Preço atualizado').reset_index(drop=True)
fig, eixos = plt.subplots(1, 2, figsize=(16, 8.2), sharey=True)
paineis = [(eixos[0], seg_o, res_o, 'Preço original', 'A) Preços originais (sem atualização)', CONTRATO),
           (eixos[1], seg_a, res_a, 'Preço atualizado', f'B) Preços atualizados pelo IPCA até {DATA_BASE}', CONTRATO_ATU)]
for ax, seg, res, col, tit, ctr in paineis:
    grupo = seg.set_index('Ref')['Grupo']
    for i, r in ordem.iterrows():
        g = int(grupo[r['Ref']])
        ax.barh(i, r[col], color=COR_GRUPO[g], edgecolor='black', lw=0.5, height=0.7)
        ax.text(max(r[col], res.loc[g - 1, 'Centroide (média)']) + 25, i, f"{brl(r[col])}  G{g}", va='center', fontsize=9, fontweight='bold', zorder=5)
    for g in range(1, K + 1):
        linhas = [i for i, r in ordem.iterrows() if int(grupo[r['Ref']]) == g]
        cen = res.loc[g - 1, 'Centroide (média)']
        ax.plot([cen, cen], [min(linhas) - 0.45, max(linhas) + 0.45], color='black', lw=2, ls='-')
    ax.axvline(ctr, color=COR_CONTRATO, ls='--', lw=1.4)
    ax.axvline(PRECO_2025, color=COR_2025, ls=':', lw=1.6)
    quadro = '\n'.join(f"G{g}: centroide {brl(res.loc[g - 1, 'Centroide (média)'])} | {int(res.loc[g - 1, 'Qtde de preços'])} preço(s) | {brl(res.loc[g - 1, 'Mínimo'])} a {brl(res.loc[g - 1, 'Máximo'])}" for g in range(1, K + 1))
    ax.text(0.5, -0.17, quadro, transform=ax.transAxes, ha='center', va='top', fontsize=9.5, family='monospace',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='#999999'))
    ax.set_title(tit, loc='left'); ax.xaxis.set_major_formatter(eixo_brl); ax.set_xlim(0, d26['Preço atualizado'].max() * 1.3)
    ax.set_xlabel('Preço do ponto de função'); ax.grid(axis='y', alpha=0)
eixos[0].set_yticks(range(len(ordem))); eixos[0].set_yticklabels([f"Ref {r['Ref']} - {apelido[r['Ref']]}" for _, r in ordem.iterrows()])
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
leg = [Patch(color=COR_GRUPO[g], label=NOME_GRUPO[g]) for g in range(1, K + 1)] + [
       Line2D([0], [0], color='black', lw=2, label='Centroide do grupo'),
       Line2D([0], [0], color=COR_CONTRATO, ls='--', label=f'Contrato 014/2022/SEPLAG: {brl(CONTRATO)}'),
       Line2D([0], [0], color=COR_2025, ls=':', label=f'Preço adotado em 2025: {brl(PRECO_2025)}')]
fig.legend(handles=leg, loc='lower center', ncol=6, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('G04 - Grupos do K-Means: cada preço, seu grupo (G1, G2, G3) e o centroide', fontweight='bold', fontsize=13)
fig.tight_layout(rect=(0, 0.1, 1, 1))
salvar(fig, 'G04_kmeans_original_x_atualizado.png')

In [9]:
# ==========================================
# CÉLULA 8: Gráfico G05 - centroide de cada grupo, original x atualizado
# ==========================================
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(K); w = 0.36
vo, va = res_o['Centroide (média)'].values, res_a['Centroide (média)'].values
b1 = ax.bar(x - w / 2, vo, w, color=COR_ORIG, label='Preços originais')
b2 = ax.bar(x + w / 2, va, w, color=COR_ATU, label=f'Preços atualizados até {DATA_BASE}')
for barras, vals, base in [(b1, vo, None), (b2, va, vo)]:
    for i, (bb, v) in enumerate(zip(barras, vals)):
        txt = brl(v) if base is None else f"{brl(v)}\n({'+' if v >= base[i] else ''}{pct((v / base[i] - 1) * 100, 1)})"
        ax.text(bb.get_x() + bb.get_width() / 2, v + 25, txt, ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels([f"{NOME_GRUPO[g]}\noriginal: {int(res_o.loc[g - 1, 'Qtde de preços'])} | atualizado: {int(res_a.loc[g - 1, 'Qtde de preços'])} preço(s)" for g in range(1, K + 1)])
ax.axhline(CONTRATO, color=COR_CONTRATO, ls='--', lw=1.3, label=f'Contrato 014/2022/SEPLAG: {brl(CONTRATO)}')
ax.axhline(PRECO_2025, color=COR_2025, ls=':', lw=1.5, label=f'Preço adotado em 2025: {brl(PRECO_2025)}')
ax.yaxis.set_major_formatter(eixo_brl); ax.set_ylim(0, max(va.max(), vo.max()) * 1.2); ax.legend(loc='upper left', fontsize=9)
ax.set_title('G05 - Centroide de cada grupo: preços originais x atualizados')
salvar(fig, 'G05_centroides_original_x_atualizado.png')

## 5. Sensibilidade: o que acontece com os grupos quando a amostra muda

| Cenário | Composição |
|---|---|
| C1 | Os 9 preços |
| C2 | Sem o preço apontado como outlier pelo IQR |
| C3 | Só os preços com até 13 meses na data da pesquisa |
| C4 | Só os preços de pregão (houve disputa de preço) |
| C5 | Os 9 preços mais o Contrato 014/2022/SEPLAG (critério de 2025, que incluiu o contrato na amostra) |

In [10]:
# ==========================================
# CÉLULA 9: Cenários de sensibilidade (preços originais e atualizados)
# ==========================================
contrato_linha = pd.DataFrame([{'Ref': 0, 'Rotulo': 'Contrato 014/2022', 'Preço original': CONTRATO, 'Preço atualizado': CONTRATO_ATU}])
cenarios = {
    'C1 - 9 preços': d26,
    'C2 - sem outlier IQR': d26[~d26['Ref'].isin(outlier_refs)],
    'C3 - até 13 meses': d26[d26['Dentro de 13 meses'] == 'Sim'],
    'C4 - só pregões': d26[d26['Disputa de preço'] == 'Sim'],
    'C5 - 9 preços + Contrato 014/2022': pd.concat([d26, contrato_linha], ignore_index=True),
}
linhas, detalhe = [], {}
for nome, d in cenarios.items():
    for base in ['Preço original', 'Preço atualizado']:
        seg, res, sil = kmeans_grupos(d, base)
        lin = {'Cenário': nome, 'Base': 'Original' if base == 'Preço original' else f'Atualizado {DATA_BASE}', 'Qtde': len(d),
               'Mediana geral': d[base].median(), 'Silhueta': round(sil, 2)}
        for g in range(1, K + 1):
            r = res.iloc[g - 1]
            lin[f'G{g} centroide'] = r['Centroide (média)']; lin[f'G{g} qtde'] = int(r['Qtde de preços'])
        linhas.append(lin)
        detalhe[f'{nome} | {lin["Base"]}'] = res.to_dict(orient='records')
tab_cen = pd.DataFrame(linhas)
mostra(tab_cen, moeda=['Mediana geral', 'G1 centroide', 'G2 centroide', 'G3 centroide'])

Cenário,Base,Qtde,Mediana geral,Silhueta,G1 centroide,G1 qtde,G2 centroide,G2 qtde,G3 centroide,G3 qtde
C1 - 9 preços,Original,9,"R$ 750,00",0.610000,"R$ 654,69",6,"R$ 1.162,11",2,"R$ 2.192,73",1
C1 - 9 preços,Atualizado 10/2026,9,"R$ 783,30",0.640000,"R$ 683,16",6,"R$ 1.215,09",2,"R$ 2.290,10",1
C2 - sem outlier IQR,Original,8,"R$ 742,50",0.770000,"R$ 552,72",3,"R$ 756,67",3,"R$ 1.162,11",2
C2 - sem outlier IQR,Atualizado 10/2026,8,"R$ 776,16",0.800000,"R$ 585,54",3,"R$ 780,78",3,"R$ 1.215,09",2
C3 - até 13 meses,Original,6,"R$ 767,50",0.520000,"R$ 702,00",4,"R$ 1.264,22",1,"R$ 2.192,73",1
C3 - até 13 meses,Atualizado 10/2026,6,"R$ 786,65",0.540000,"R$ 725,13",4,"R$ 1.316,01",1,"R$ 2.290,10",1
C4 - só pregões,Original,5,"R$ 735,00",0.560000,"R$ 546,50",2,"R$ 735,00",1,"R$ 1.162,11",2
C4 - só pregões,Atualizado 10/2026,5,"R$ 769,02",0.520000,"R$ 579,81",2,"R$ 769,02",1,"R$ 1.215,09",2
C5 - 9 preços + Contrato 014/2022,Original,10,"R$ 742,50",0.640000,"R$ 638,95",7,"R$ 1.162,11",2,"R$ 2.192,73",1
C5 - 9 preços + Contrato 014/2022,Atualizado 10/2026,10,"R$ 776,16",0.650000,"R$ 663,35",7,"R$ 1.215,09",2,"R$ 2.290,10",1


## 6. Comparação 2025 x 2026

Para comparar valores de datas diferentes, usam-se três leituras:

* **Original:** cada ano com seus preços sem atualização.
* **Atualizado em cada ano:** 2025 até 10/2025 (como publicado no inventário de 2025) e 2026 até a data-base atual.
* **Mesma data-base:** os preços de 2025 (já atualizados até 10/2025) são corrigidos pelo IPCA de 11/2025 até a data-base de 2026. É a comparação justa, com a inflação neutralizada.

Os grupos de 2025 com preços atualizados são recalculados aqui com o mesmo algoritmo e reproduzem exatamente os valores publicados (R$ 633,92, R$ 943,25 e R$ 1.174,50).
Em 2025 não foi feito K-Means sobre os preços originais; esse agrupamento é calculado aqui só para a comparação.

In [11]:
# ==========================================
# CÉLULA 10: K-Means de 2025 (recalculado) e comparação com 2026
# ==========================================
col_base = f'Atualizado até {DATA_BASE}'
s25o, r25o, _ = kmeans_grupos(d25, 'Preço original')
s25a, r25a, _ = kmeans_grupos(d25, 'Atualizado até 10/2025')
s25b, r25b, _ = kmeans_grupos(d25, col_base)
publicado_2025 = [633.92, 943.25, 1174.50]
assert np.allclose(r25a['Centroide (média)'].round(2).values, publicado_2025), 'O K-Means não reproduziu os grupos publicados em 2025'
print('Conferência: os grupos de 2025 foram reproduzidos exatamente.')

comp = pd.DataFrame({
    'Grupo': [NOME_GRUPO[g] for g in range(1, K + 1)],
    '2025 original': r25o['Centroide (média)'].values,
    '2026 original': res_o['Centroide (média)'].values,
    '2025 atualizado até 10/2025 (publicado)': r25a['Centroide (média)'].values,
    f'2025 atualizado até {DATA_BASE}': r25b['Centroide (média)'].values,
    f'2026 atualizado até {DATA_BASE}': res_a['Centroide (média)'].values,
})
comp['Variação original (2026 / 2025)'] = (comp['2026 original'] / comp['2025 original'] - 1) * 100
comp[f'Variação real, mesma data-base {DATA_BASE}'] = (comp[f'2026 atualizado até {DATA_BASE}'] / comp[f'2025 atualizado até {DATA_BASE}'] - 1) * 100
mostra(comp, moeda=[c for c in comp.columns if c.startswith('202')], pcts=[c for c in comp.columns if c.startswith('Variação')])

print('\nGrupos de 2025 com os preços atualizados até 10/2025 (como publicado):'); mostra(r25a, moeda=moeda)
print(f'\nGrupos de 2025 trazidos até {DATA_BASE}:'); mostra(r25b, moeda=moeda)
print('\nFontes que aparecem nos dois anos: Casa da Moeda (PE 90061/2024) e Marinha (PE 90002/2025).')
print(' - Marinha: R$ 1.060,00 nos dois anos (2025: Banco de Preços, menor preço; 2026: Ref 3).')
print(' - Casa da Moeda: 2025 usou R$ 1.182,00, média saneada das propostas finais; 2026 usa R$ 555,00, preço homologado do item 1 (Ref 1).')

Conferência: os grupos de 2025 foram reproduzidos exatamente.


Grupo,2025 original,2026 original,2025 atualizado até 10/2025 (publicado),2025 atualizado até 10/2026,2026 atualizado até 10/2026,Variação original (2026 / 2025),"Variação real, mesma data-base 10/2026"
Grupo 1 (Entrada),"R$ 608,74","R$ 654,69","R$ 633,92","R$ 663,25","R$ 683,16","7,55%","3,00%"
Grupo 2 (Intermediário),"R$ 900,70","R$ 1.162,11","R$ 943,25","R$ 986,91","R$ 1.215,09","29,02%","23,12%"
Grupo 3 (Premium),"R$ 1.121,00","R$ 2.192,73","R$ 1.174,50","R$ 1.228,86","R$ 2.290,10","95,60%","86,36%"



Grupos de 2025 com os preços atualizados até 10/2025 (como publicado):


Grupo,Qtde de preços,Preços do grupo,Mínimo,Mediana,Centroide (média),Máximo
Grupo 1 (Entrada),7,"P10: R$ 523,74; P2: R$ 584,76; P3: R$ 612,60; P5: R$ 629,48; P4: R$ 668,29; P13: R$ 677,78; P8: R$ 740,76","R$ 523,74","R$ 629,48","R$ 633,92","R$ 740,76"
Grupo 2 (Intermediário),4,"P7: R$ 924,40; P9: R$ 928,01; P6: R$ 933,20; P1: R$ 987,41","R$ 924,40","R$ 930,61","R$ 943,25","R$ 987,41"
Grupo 3 (Premium),2,"P11: R$ 1.105,00; P12: R$ 1.244,00","R$ 1.105,00","R$ 1.174,50","R$ 1.174,50","R$ 1.244,00"



Grupos de 2025 trazidos até 10/2026:


Grupo,Qtde de preços,Preços do grupo,Mínimo,Mediana,Centroide (média),Máximo
Grupo 1 (Entrada),7,"P10: R$ 547,98; P2: R$ 611,82; P3: R$ 640,95; P5: R$ 658,61; P4: R$ 699,22; P13: R$ 709,15; P8: R$ 775,05","R$ 547,98","R$ 658,61","R$ 663,25","R$ 775,05"
Grupo 2 (Intermediário),4,"P7: R$ 967,18; P9: R$ 970,96; P6: R$ 976,39; P1: R$ 1.033,11","R$ 967,18","R$ 973,67","R$ 986,91","R$ 1.033,11"
Grupo 3 (Premium),2,"P11: R$ 1.156,14; P12: R$ 1.301,58","R$ 1.156,14","R$ 1.228,86","R$ 1.228,86","R$ 1.301,58"



Fontes que aparecem nos dois anos: Casa da Moeda (PE 90061/2024) e Marinha (PE 90002/2025).
 - Marinha: R$ 1.060,00 nos dois anos (2025: Banco de Preços, menor preço; 2026: Ref 3).
 - Casa da Moeda: 2025 usou R$ 1.182,00, média saneada das propostas finais; 2026 usa R$ 555,00, preço homologado do item 1 (Ref 1).


In [12]:
# ==========================================
# CÉLULA 11: Gráfico G06 - todos os preços de 2025 e 2026 na mesma régua
# ==========================================
faixas_ano = [('2025 original', s25o, 'Preço original'), ('2026 original', seg_o, 'Preço original'),
              (f'2025 atualizado até {DATA_BASE}', s25b, col_base), (f'2026 atualizado até {DATA_BASE}', seg_a, 'Preço atualizado')]
fig, ax = plt.subplots(figsize=(14, 6.5))
for y, (nome, seg, col) in enumerate(faixas_ano[::-1]):
    for g in range(1, K + 1):
        x = seg[seg['Grupo'] == g]
        ax.scatter(x[col], [y] * len(x), s=95, color=COR_GRUPO[g], edgecolor='black', zorder=3, alpha=0.9,
                   label=NOME_GRUPO[g] if y == 0 else None)
        cen = x[col].mean()
        ax.plot([cen, cen], [y - 0.3, y + 0.3], color=COR_GRUPO[g], lw=3)
        ax.text(cen, y + 0.33, brl(cen, 0), ha='center', fontsize=8, fontweight='bold', color=COR_GRUPO[g])
ax.set_yticks(range(len(faixas_ano))); ax.set_yticklabels([f[0] for f in faixas_ano[::-1]])
ax.axhline(1.5, color='black', lw=0.8)
ax.axvline(CONTRATO, color=COR_CONTRATO, ls='--', lw=1.2, label=f'Contrato 014/2022: {brl(CONTRATO)}')
ax.axvline(PRECO_2025, color=COR_2025, ls=':', lw=1.4, label=f'Preço adotado em 2025: {brl(PRECO_2025)}')
ax.xaxis.set_major_formatter(eixo_brl); ax.set_xlabel('Preço do ponto de função  (pontos = preços; traço = centroide do grupo)')
ax.set_ylim(-0.6, len(faixas_ano) - 0.3); ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=5, fontsize=9)
ax.set_title('G06 - Preços e grupos de 2025 e de 2026 na mesma régua')
salvar(fig, 'G06_precos_2025_x_2026.png')

In [13]:
# ==========================================
# CÉLULA 12: Gráfico G07 - centroides 2025 x 2026 (original e mesma data-base)
# ==========================================
fig, eixos = plt.subplots(1, 2, figsize=(14, 5.2), sharey=True)
for ax, c25, c26, tit in [(eixos[0], '2025 original', '2026 original', 'A) Preços originais'),
                          (eixos[1], f'2025 atualizado até {DATA_BASE}', f'2026 atualizado até {DATA_BASE}', f'B) Mesma data-base ({DATA_BASE})')]:
    x = np.arange(K); w = 0.36
    for desl, col, cor in [(-w / 2, c25, COR_2025), (w / 2, c26, COR_ATU)]:
        bb = ax.bar(x + desl, comp[col], w, color=cor, label=col.split(' ')[0])
        for b, v in zip(bb, comp[col]):
            ax.text(b.get_x() + b.get_width() / 2, v + 20, brl(v), ha='center', fontsize=8.5, fontweight='bold')
    for i in range(K):
        var = (comp[c26][i] / comp[c25][i] - 1) * 100
        ax.text(i, max(comp[c25][i], comp[c26][i]) + 140, f"{'+' if var >= 0 else ''}{pct(var, 1)}", ha='center', fontsize=10,
                color='#c62828' if var < 0 else '#2e7d32', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels([NOME_GRUPO[g] for g in range(1, K + 1)]); ax.set_title(tit, loc='left')
    ax.yaxis.set_major_formatter(eixo_brl); ax.legend(loc='upper left')
eixos[0].set_ylim(0, comp[[c for c in comp.columns if c.startswith('202')]].values.max() * 1.25)
fig.suptitle('G07 - Centroide de cada grupo: 2025 x 2026 (variação em cima das barras)', fontweight='bold')
fig.tight_layout()
salvar(fig, 'G07_centroides_2025_x_2026.png')

### Gráfico de teia (radar)

Cada eixo é uma referência de preço; a distância ao centro é o preço. Mostra de uma vez o preço original, o atualizado e as linhas de referência (contrato e preço de 2025).
Atenção na leitura: a área desenhada depende da ordem dos eixos e o preço mais alto comprime os demais. Por isso o painel B repete o gráfico sem o preço apontado como outlier. O gráfico complementa o G04, não o substitui.

In [14]:
# ==========================================
# CÉLULA 12b: Gráfico G08 - teia (radar) dos preços por referência
# ==========================================
def teia(ax, dados, titulo):
    n = len(dados)
    ang = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    fecha = lambda v: list(v) + [v[0]]
    angf = ang + ang[:1]
    for col, cor, nome in [('Preço original', COR_ORIG, 'Preço original'), ('Preço atualizado', COR_ATU, f'Preço atualizado até {DATA_BASE}')]:
        ax.plot(angf, fecha(dados[col]), color=cor, lw=2, label=nome)
        ax.fill(angf, fecha(dados[col]), color=cor, alpha=0.12)
        ax.scatter(ang, dados[col], color=cor, s=25, zorder=3)
    for v, cor, ls, nome in [(CONTRATO, COR_CONTRATO, '--', f'Contrato 014/2022/SEPLAG: {brl(CONTRATO)}'), (PRECO_2025, COR_2025, ':', f'Preço adotado em 2025: {brl(PRECO_2025)}')]:
        ax.plot(angf, [v] * (n + 1), color=cor, ls=ls, lw=1.6, label=nome)
    ax.set_xticks(ang)
    ax.set_xticklabels([f"Ref {r['Ref']}\n{apelido[r['Ref']]}\n{brl(r['Preço atualizado'], 0)}" for _, r in dados.iterrows()], fontsize=8.5)
    ax.set_ylim(0, dados['Preço atualizado'].max() * 1.08)
    ax.yaxis.set_major_formatter(eixo_brl); ax.tick_params(axis='y', labelsize=7.5)
    ax.set_title(titulo, pad=28, fontsize=11)

fig, eixos = plt.subplots(1, 2, figsize=(16, 8), subplot_kw={'projection': 'polar'})
base_teia = d26.sort_values('Ref').reset_index(drop=True)
teia(eixos[0], base_teia, 'A) Os 9 preços')
teia(eixos[1], base_teia[~base_teia['Ref'].isin(outlier_refs)].reset_index(drop=True), 'B) Sem o preço apontado como outlier (escala ampliada)')
h, l = eixos[0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=4, fontsize=9.5, bbox_to_anchor=(0.5, -0.01))
fig.suptitle('G08 - Gráfico de teia: preço de cada referência, original x atualizado', fontweight='bold', fontsize=13)
fig.tight_layout(rect=(0, 0.06, 1, 0.92))
salvar(fig, 'G08_teia_precos_por_referencia.png')

## 7. Gravação dos resultados

Os resultados vão para `resultados_analise.json` (lido pela manifestação técnica) e `Resultados_Analise_PF_2026.xlsx` (para consulta).

In [15]:
# ==========================================
# CÉLULA 13: Exporta resultados
# ==========================================
def registros(df):
    return json.loads(df.to_json(orient='records', force_ascii=False))
resultado = {
    'data_base_ipca': DATA_BASE,
    'contrato': {'preco_vigente': CONTRATO, 'preco_na_analise_atualizada': CONTRATO_ATU, 'mes_base_informado': ct.get('mes_base_preco_vigente')},
    'preco_2025': PRECO_2025,
    'fator_ipca_2025_para_base': FATOR_2025_2026,
    'amostra_2026': registros(d26),
    'descritiva': desc,
    'outlier_refs': outlier_refs,
    'kmeans_2026_original': {'silhueta': sil_o, 'grupos': registros(res_o), 'atribuicao': dict(zip(seg_o['Ref'].astype(str), seg_o['Grupo']))},
    'kmeans_2026_atualizado': {'silhueta': sil_a, 'grupos': registros(res_a), 'atribuicao': dict(zip(seg_a['Ref'].astype(str), seg_a['Grupo']))},
    'cenarios': registros(tab_cen),
    'cenarios_detalhe': detalhe,
    'amostra_2025': registros(d25),
    'kmeans_2025_original': registros(r25o), 'kmeans_2025_atualizado_10_2025': registros(r25a), 'kmeans_2025_atualizado_base': registros(r25b),
    'comparacao_2025_2026': registros(comp),
}
(PASTA / 'resultados_analise.json').write_text(json.dumps(resultado, ensure_ascii=False, indent=1, default=float), encoding='utf-8')
with pd.ExcelWriter(PASTA / 'Resultados_Analise_PF_2026.xlsx') as xw:
    d26.to_excel(xw, sheet_name='Amostra 2026', index=False)
    tab_desc.drop(index='Outliers').to_excel(xw, sheet_name='Descritiva')
    tab_ipca.to_excel(xw, sheet_name='Atualizacao IPCA', index=False)
    res_o.to_excel(xw, sheet_name='Grupos original', index=False)
    res_a.to_excel(xw, sheet_name='Grupos atualizado', index=False)
    tab_cen.to_excel(xw, sheet_name='Cenarios', index=False)
    d25.to_excel(xw, sheet_name='Amostra 2025', index=False)
    comp.to_excel(xw, sheet_name='Comparacao 2025 x 2026', index=False)
print('Gravados: resultados_analise.json, Resultados_Analise_PF_2026.xlsx e gráficos em', GRAF)

Gravados: resultados_analise.json, Resultados_Analise_PF_2026.xlsx e gráficos em /home/user/Seplag/Inventário de Ativos Intangíveis/2026/2. Levantamento_Identificacao_Avaliacao/2.3 Mapa Comparativo de Preços/5. Análise Estatística/graficos
